# Two interpreters, one number, and a gate that says so

A sampler wants a fast, differentiable log density. A diagnostic wants a readable one. The
usual resolution is to write both: numpy for clarity, jax for speed, and a comment promising
they agree. They agree until someone fixes a bug in one of them.

A backend here fits a *model spec*, not a Python callable (plan review A2): the mean
expression, the outcome column, a likelihood, and a prior for every parameter. The same tree
is evaluated by numpy (`value`, `log_density`) and by jax (`compile_jax`,
`compile_log_density`), and the two **must** agree to eight decimals on every shipped model —
that agreement is gate 9, the numerical replacement for "both call `forward()`".

Three nodes support panel models: `Reduce` (sum/mean/max along the last axis, used to
normalize carryover weights), `Gather` (a vector parameter indexed by an integer column — a
unit-level intercept), and a vector `Const` (a lag index).

In [ ]:
import numpy as np

from axiom.core import (
    PRIOR_HYPER, Add, Const, Convolve, D, Data, DesignMatrix, Div, Gather, Likelihood, LikelihoodFamily,
    ModelSpec, Mul, Param, Pow, Prior, PriorFamily, Reduce, Spec, compile_jax, compile_log_density,
    constrain, dimension, dimensionless, free_parameters, jax_available, log_density, log_prior,
    require_jax, unconstrain, value,
)

from axiom.display import enable

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import BLUE, ORANGE, annotate, caption, compare, lines, mark_x, scatter_fit

enable();  # every axiom result renders itself from here on

## Priors and hierarchy

`PRIOR_HYPER` lists the hyperparameters each family takes. A hyperparameter may *name another
parameter* — that is how a hierarchy is declared, with no separate "hierarchical model" class
and no place for the pooling structure to disagree with the priors.

In [ ]:
print(PRIOR_HYPER)
fam: PriorFamily = "halfnormal"
a_mean = Param(name="a_mean", dimension=D.outcome, prior=Prior(family="normal", hyper={"mu": 0.0, "sigma": 10.0}))
a_sd = Param(name="a_sd", dimension=D.outcome, prior=Prior(family=fam, hyper={"sigma": 5.0}))
alpha = Param(name="alpha", dimension=D.outcome, shape=(3,), prior=Prior(family="normal", hyper={"mu": "a_mean", "sigma": "a_sd"}))
print(alpha.prior.parents, alpha.size)

## Carryover weights as a pure expression

No opaque function is needed: `Pow(lam, lags) / Reduce(sum)` is the normalized geometric
weight vector, and `Convolve` applies it causally. The jax interpreter therefore covers it —
an opaque Python kernel would have been the one part of the model that could not be
differentiated, which is to say the one part a sampler would be slowest on.

In [ ]:
lam = Param(name="lam", dimension=dimensionless(), prior=Prior(family="beta", hyper={"alpha": 2.0, "beta": 2.0}))
lags = Const(value=tuple(float(i) for i in range(6)), dimension=dimensionless())
raw = Pow(base=lam, exponent=lags)
weights = Div(numerator=raw, denominator=Reduce(op="sum", arg=raw))
dose = Data(name="dose", dimension=D.currency)
carried = Convolve(signal=dose, kernel=weights)
impulse = np.array([100.0, 0, 0, 0, 0, 0, 0, 0])
print(dimension(carried), value(carried, data={"dose": impulse}, params={"lam": 0.5}).round(3))

In [ ]:
decay = {f"λ = {lv}": value(carried, data={"dose": impulse}, params={"lam": lv}) for lv in (0.2, 0.5, 0.8)}
fig = lines(
    np.arange(len(impulse)), decay,
    title="One parameter, the whole memory of the system",
    subtitle="response to 100 units of dose in period 0, weights normalized to sum to one",
    x_title="period", y_title="carried dose",
)
caption(fig, "Normalization is what makes λ a memory parameter rather than a second gain: "
             "the area under each curve is the same 100 units, moved around in time.")

## A hierarchical panel model

In [ ]:
unit = Data(name="unit", dimension=dimensionless())
y = Data(name="y", dimension=D.outcome)
k = Param(name="k", dimension=D.currency, prior=Prior(family="lognormal", hyper={"mu": float(np.log(50)), "sigma": 0.5}))
s = Param(name="s", dimension=dimensionless(), prior=Prior(family="gamma", hyper={"alpha": 4.0, "beta": 2.0}))
beta = Param(name="beta", dimension=D.outcome, prior=Prior(family="halfnormal", hyper={"sigma": 20.0}))
sigma = Param(name="sigma", dimension=D.outcome, prior=Prior(family="halfnormal", hyper={"sigma": 5.0}))
u = Div(numerator=carried, denominator=k)
hill = Mul(factors=(beta, Div(numerator=Pow(base=u, exponent=s), denominator=Add(terms=(Const(value=1.0, dimension=dimensionless()), Pow(base=u, exponent=s))))))
mean = Add(terms=(Gather(source=alpha, index=unit), hill))

family: LikelihoodFamily = "normal"
model = ModelSpec(
    name="panel_hill",
    mean=mean,
    outcome=y,
    likelihood=Likelihood(family=family, scale="sigma"),
    parameters=(k, s, beta, lam, a_mean, a_sd, alpha, sigma),
)
print([p.name for p in free_parameters(model)])
print("shapes:", [p.name for p in model.shapes], "| scales:", [p.name for p in model.scales])
print(Spec.from_json(model.to_json()) == model)

## Unconstrained coordinates and the log density

Inference happens on $\mathbb{R}^k$: `unconstrain` maps each parameter by its support (log for
positive, logit for the unit interval), `constrain` maps back and returns the log-Jacobian, and
`log_density` = log prior + Jacobian + log likelihood. Forgetting the Jacobian is the classic
silent error here: the sampler still runs, the chains still mix, and the posterior is wrong by
a factor that depends on where it is.

In [ ]:
rng = np.random.default_rng(0)
n = 24
data = {"unit": rng.integers(0, 3, n), "dose": rng.uniform(0, 150, n)}
theta = {"k": 50.0, "s": 2.0, "beta": 10.0, "lam": 0.5, "a_mean": 1.0, "a_sd": 0.5, "alpha": np.array([0.5, 1.0, 1.5]), "sigma": 1.0}
data["y"] = value(mean, data=data, params=theta) + rng.normal(0, 1, n)

z = unconstrain(model, theta)
back, log_jac = constrain(model, z)
print({k_: np.round(v, 3) for k_, v in z.items() if k_ in ("k", "lam", "s")}, "| log|J| =", round(log_jac, 3))
print("round-trips:", all(np.allclose(back[k_], theta[k_]) for k_ in theta))
print("log prior:", round(log_prior(model, theta), 3), "| log density:", round(log_density(model, data, z), 3))

## The jax interpreter

`compile_jax` turns the tree into a traceable function; `compile_log_density` does the same for
the whole posterior. When jax is not installed, `require_jax()` returns a typed `Unsupported`
instead of an `ImportError`.

In [ ]:
print("jax available:", jax_available(), "|", require_jax())
if jax_available():
    import jax

    jax.config.update("jax_enable_x64", True)
    f_mean = compile_jax(mean)
    f_ld = compile_log_density(model)
    print("mean agrees:", np.allclose(np.asarray(f_mean(data, theta)), value(mean, data=data, params=theta)))
    print("log density agrees:", abs(float(f_ld(data, z)) - log_density(model, data, z)) < 1e-8)
    grad = jax.grad(lambda zz: f_ld(data, zz))(z)
    print({k_: np.asarray(v).shape for k_, v in grad.items()})

In [ ]:
# Gate 9, drawn: the two interpreters over forty random points of parameter space.
points = []
for i in range(40):
    jitter = {p: v * float(np.exp(rng.normal(0, 0.25))) if np.ndim(v) == 0 else v + rng.normal(0, 0.2, np.shape(v))
              for p, v in theta.items()}
    jitter["lam"] = float(np.clip(jitter["lam"], 0.05, 0.95))
    zz = unconstrain(model, jitter)
    numpy_ld = log_density(model, data, zz)
    jax_ld = float(compile_log_density(model)(data, zz)) if jax_available() else numpy_ld
    points.append((numpy_ld, jax_ld))

fig = scatter_fit(
    [p[0] for p in points], [p[1] for p in points],
    title="Gate 9, drawn",
    subtitle="log density under the numpy interpreter against the jax one, forty random points",
    x_title="numpy log density", y_title="jax log density",
)
worst = max(abs(a - b) for a, b in points)
caption(fig, f"Largest disagreement across the forty points: {worst:.2e}. The gate fails the "
             f"build at 1e-8, which is why 'the fast path' cannot quietly become a different model.")

## A known per-row scale: `Likelihood.scale_expr`

A continuous likelihood takes *either* a scale parameter name (`scale="sigma"`, as above) *or* a
`scale_expr`: an expression over data and parameters giving the scale per row. That is how a
meta-analysis declares a known per-study standard error (`Data("se")`) without inventing a
parameter for it — a parameter that would then be estimated, from the very studies whose
precision it was supposed to encode.

In [ ]:
from axiom.core.model import likelihood_scale

se_col = Data(name="se", dimension=D.outcome)
mu_p = Param(name="mu", dimension=D.outcome, prior=Prior(family="normal", hyper={"mu": 0.0, "sigma": 10.0}))
known_scale = ModelSpec(
    name="known_se",
    mean=mu_p,
    outcome=y,
    likelihood=Likelihood(family="normal", scale_expr=se_col),
    parameters=(mu_p,),
)
print("data columns:", known_scale.data_columns, "| scales:", [p.name for p in known_scale.scales])
print("scale per row:", likelihood_scale(known_scale, {"se": np.array([0.1, 0.2, 0.4])}, {"mu": 1.0}))
try:
    Likelihood(family="normal", scale="sigma", scale_expr=se_col)
except ValueError as e:
    print("either/or:", e)

In [ ]:
from axiom.core import log_likelihood, log_prior

# log_density = log_prior + log_likelihood (+ the Jacobian of the unconstraining map)
print('decomposition available:', callable(log_likelihood), callable(log_prior))

## Soft constraints: `Constraint`

A `ModelSpec` may also carry **soft constraints** (decision D6.3): a scalar expression over the
model's parameters and data — typically an estimand realized at an experiment's doses and reduced
over its units and periods — with an observed value and a scale. Each adds
`log p(observed | expr(θ), scale)` to the likelihood: `family="normal"` contributes
`log N(observed | expr, scale)`; `family="lognormal"` contributes a multiplicative error with
`scale` on the log scale and is `-inf` wherever `expr` is not positive (never an exception inside
the density). This is how `axiom.calibrate` folds a randomized measurement into the graph without
a second model. The jax interpreter adds the identical term — gate 9 checks every shipped
`ModelSpec`, constraints included.

In [ ]:
from scipy import stats

from axiom.core import Constraint

lift = Constraint(
    name="mean_lift@experiment_1",
    expr=Reduce(op="mean", arg=hill),
    family="normal",
    observed=6.0,
    scale=0.5,
    detail={"measurement": "experiment_1", "doses": "observed"},
)
constrained = ModelSpec(
    name="panel_hill_constrained",
    mean=mean,
    outcome=y,
    likelihood=Likelihood(family=family, scale="sigma"),
    parameters=model.parameters,
    constraints=(lift,),
)
E = float(value(lift.expr, data=data, params=theta))
delta = log_density(constrained, data, z) - log_density(model, data, z)
print("expr(theta) =", round(E, 4), "| added term:", round(delta, 6), "= log N(6.0 | expr, 0.5):", round(float(stats.norm.logpdf(6.0, E, 0.5)), 6))
print("round-trips:", Spec.from_json(constrained.to_json()) == constrained)
if jax_available():
    print("jax agrees:", abs(float(compile_log_density(constrained)(data, z)) - log_density(constrained, data, z)) < 1e-8)

In [ ]:
betas = np.linspace(4.0, 18.0, 40)
plain, with_lift = [], []
for b in betas:
    zz = unconstrain(model, {**theta, "beta": float(b)})
    plain.append(log_density(model, data, zz))
    with_lift.append(log_density(constrained, data, zz))
plain = np.array(plain) - max(plain)
with_lift = np.array(with_lift) - max(with_lift)

fig = lines(
    betas,
    {"observational panel only": plain, "panel + the experiment": with_lift},
    colors=(BLUE, ORANGE),
    title="What one randomized measurement does to a curve",
    subtitle="log density along beta, each normalized to its own maximum — the constraint is the experiment",
    x_title="beta", y_title="log density (relative)",
)
mark_x(fig, float(betas[int(np.argmax(with_lift))]), text="mode with the experiment")
caption(fig, "The measurement is not a separate model whose answer gets averaged in "
             "afterwards. It is a term in this density, so everything correlated with beta "
             "moves with it — which is the whole argument for calibrating rather than blending.")

## `DesignMatrix`

`SupportsForward.linearize` returns a `DesignMatrix`: at a fixed point of the nonlinear
parameters, the mean is `offset + X @ theta[columns]`. Its invariant against `forward()` is
gate 9's other half and lands with `surface.linearize` — every closed-form design calculation
in `axiom.design` runs on this, so it had better be the same model.

In [ ]:
X = np.column_stack([np.ones(4), np.arange(4.0)])
dm = DesignMatrix(X=X, columns=("a", "b"), offset=np.zeros(4), at={"k": np.array(50.0)})
print(dm.predict({"a": 1.0, "b": 2.0}))

## Seeing it

`enable()` at the top of this notebook already made a bare result on the last
line of a cell render itself — a card drawn by `rich`, or the same content as
aligned plain text where `rich` is not installed. `show` does it on demand, for
a result that is not the last thing in its cell.

`axiom.viz` draws the figure this subpackage's results are actually about.

In [ ]:
from axiom.core import Interval, Verdict
from axiom.display import show
from axiom.viz import intervals

show(Verdict(status="identified", reason="age blocks the only back-door path", route="backdoor"))

doses = {
    "40 mg": Interval(lower=-16.7, upper=-8.1, definition="eti", mass=0.9),
    "20 mg": Interval(lower=-6.2, upper=-3.4, definition="eti", mass=0.9),
    "10 mg": Interval(lower=-2.1, upper=1.4, definition="eti", mass=0.9),
}
intervals(doses, unit="mmHg")

The 10 mg band crosses zero and the other two do not. That is a fact about six numbers, and
it is one glance on a shared scale against three separate comparisons in a printed list —
which is the whole argument for drawing intervals rather than reporting them.

## What this bought you

A model that a sampler can differentiate, a diagnostic can read, and a reviewer can typeset,
with a gate standing between the fast version and the readable one. Priors that declare a
hierarchy without a second class to hold it. And a way to fold a randomized measurement into
the density itself rather than reconciling two answers afterwards.

`nbs/infer/01-backends.ipynb` hands this spec to three samplers; `nbs/surface/` fits it to a
panel; `nbs/calibrate/03-likelihood-route.ipynb` is that constraint in anger.

## Which likelihood, and what it costs you to change one

`Likelihood` names a family. Six are available, and the choice is not only
about the density: it decides how much a single observation is *worth*, which
is what every design calculation in `axiom.design` is ultimately asking.

For any of these families the Fisher information has the same shape,

$$\mathcal{I} = J^\top W J, \qquad J = \frac{\partial\,\text{mean}}{\partial\theta},
\qquad w_i = \frac{1}{\phi\,V(\mu_i)}$$

so one diagonal weight per row is the whole difference between a Gaussian
design and a Poisson one. `variance_weight` is that weight.

In [ ]:
import numpy as np

from axiom.core import variance_weight

mu = np.array([0.5, 2.0, 8.0])
for family, kw in [
    ("normal", {"scale": 2.0}),
    ("poisson", {}),
    ("gamma", {"scale": 0.5}),
    ("lognormal", {"scale": 0.5}),
    ("student_t", {"scale": 2.0, "df": 4.0}),
]:
    w = np.broadcast_to(variance_weight(family, mu, **kw), mu.shape)
    print(f"{family:10s} w = {np.array2string(w, precision=4)}")

Read the rows against each other. The `normal` weight ignores the mean
entirely — that is what homoscedastic means, and it is why a Gaussian design
can be computed once and reused anywhere on the surface. Every other row falls
as the mean rises: a Poisson observation at $\mu = 8$ carries a sixteenth the
information about the mean that one at $\mu = 0.5$ does, because its variance
grew with it. A design that puts all its rows where the response is large is
buying much less than the Gaussian arithmetic would tell you.

`student_t` is not an exponential family, but its location information is a
constant multiple of the Gaussian one, so it fits here anyway — and the
multiple tends to 1 as the tails thin out:

In [ ]:
normal = float(variance_weight("normal", 1.0, scale=2.0))
for df in (4.0, 10.0, 100.0, 10_000.0):
    ratio = float(variance_weight("student_t", 1.0, scale=2.0, df=df)) / normal
    print(f"df = {df:>8.0f}   information relative to a normal: {ratio:.4f}")

A $t$ with four degrees of freedom is worth about 71% of a normal
observation of the same scale. That is the price of the robustness, stated as
a number.

### On a fitted model

`information_weight` is the same question asked of a `ModelSpec`: it evaluates
the mean at `theta` and applies that family's variance function, so it is the
weight a *design* on this model would use.

In [ ]:
from axiom.core import (
    Apply,
    Data,
    Likelihood,
    ModelSpec,
    Param,
    Prior,
    binomial_trials,
    dimensionless,
    information_weight,
)

NONE = dimensionless()
x = Data(name="x", dimension=NONE)
b = Param(name="b", dimension=NONE, prior=Prior(family="normal", hyper={"mu": 0.0, "sigma": 1.0}))

# a binomial model: the mean expression is the success PROBABILITY, and the
# outcome column counts successes out of `trials`.
clicks = ModelSpec(
    name="clicks",
    mean=Apply(fn="sigmoid", arg=b * x),
    outcome=Data(name="y", dimension=NONE),
    likelihood=Likelihood(family="binomial", trials="n"),
    parameters=(b,),
)

data = {"x": np.array([-2.0, 0.0, 2.0]), "n": np.array([100.0, 100.0, 100.0]), "y": np.zeros(3)}
theta = {"b": 1.0}
print("trials      ", binomial_trials(clicks, data))
print("probability ", np.round(1 / (1 + np.exp(-theta["b"] * data["x"])), 4))
print("weight      ", np.round(information_weight(clicks, data, theta), 2))

The weight is smallest in the middle — at $p = 0.5$, where $p(1-p)$ is
largest and a single trial is least informative about where $p$ sits. It is a
useful thing to have in front of you before choosing doses: for a binomial
outcome the *least* informative place to measure is the one where the response
is most uncertain, which is the opposite of the intuition a Gaussian model
trains.

`binomial_trials` is the accessor the density and the weight both go through.
It refuses a family that has no trials, so a Poisson model cannot be quietly
read as a one-trial binomial:

In [ ]:
poisson = clicks.model_copy(update={"likelihood": Likelihood(family="poisson")})
try:
    binomial_trials(poisson, data)
except ValueError as e:
    print("refused:", e)